In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import joblib
import os

print("--- ArthSetu Model Training Pipeline ---")
data_path = '../data/processed/clean_features.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"❌ ERROR: {data_path} missing.")

df = pd.read_csv(data_path)

# Split
X = df.drop('Default', axis=1)
y = df['Default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("--- Balancing with SMOTE ---")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("--- Training Upgraded XGBoost Engine ---")
# Replaced default parameters with a robust configuration to avoid overfitting
model = XGBClassifier(
    n_estimators=150, 
    max_depth=5, 
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)
model.fit(X_train_balanced, y_train_balanced)

# Export
artifact_path = '../backend/models/arthsetu_xgb.pkl'
os.makedirs(os.path.dirname(artifact_path), exist_ok=True)
joblib.dump(model, artifact_path)
print(f"✅ SUCCESS: Model exported to {artifact_path}")

# ==========================================
# PERFORMANCE EVALUATION & THRESHOLD MOVING
# ==========================================
print("\n" + "="*50)
print("   UNDERWRITING PERFORMANCE (WITH THRESHOLD MOVING)")
print("="*50)

# Get raw probabilities instead of binary 0/1 classes
y_pred_proba = model.predict_proba(X_test)[:, 1] 

# Apply Custom Threshold (The Business Dial)
# Instead of 0.50, we use 0.40 to be more aggressive against risky borrowers
CUSTOM_THRESHOLD = 0.40
y_pred_custom = (y_pred_proba >= CUSTOM_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f} (Independent of threshold)")

accuracy = accuracy_score(y_test, y_pred_custom)
print(f"Accuracy at {CUSTOM_THRESHOLD} threshold: {accuracy * 100:.2f}%\n")

print(f"Confusion Matrix (Actual vs Predicted at {CUSTOM_THRESHOLD} threshold):")
cm = confusion_matrix(y_test, y_pred_custom)
print(f"True Negatives:  {cm[0][0]:<6} (Approved correctly)")
print(f"False Positives: {cm[0][1]:<6} (Rejected unfairly - Acceptable loss)")
print(f"False Negatives: {cm[1][0]:<6} (Approved wrongly - DANGEROUS)")
print(f"True Positives:  {cm[1][1]:<6} (Rejected correctly)")
print("="*50)

--- ArthSetu Model Training Pipeline ---
Loading clean data...
Balancing classes with SMOTE...
Training XGBoost Engine...


/Users/bijaya/arthsetu_project/backend/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [02:08:25] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ SUCCESS: Model exported to ../backend/models/arthsetu_xgb.pkl

       UNDERWRITING PERFORMANCE REPORT
Model Accuracy: 92.12%
ROC-AUC Score:  0.8319

Detailed Metrics:
               precision    recall  f1-score   support

Good Standing       0.96      0.96      0.96     27995
      Default       0.41      0.40      0.40      2005

     accuracy                           0.92     30000
    macro avg       0.68      0.68      0.68     30000
 weighted avg       0.92      0.92      0.92     30000


Confusion Matrix (Actual vs Predicted):
True Negatives:  26835 (Correctly predicted Paybacks)
False Positives: 1160 (Wrongly predicted Defaults)
False Negatives: 1205 (Wrongly predicted Paybacks)
True Positives:  800 (Correctly predicted Defaults)
